# 📖 AudiobookMaker Kaggle WebUI

Welcome to the Kaggle Notebook for **AudiobookMaker** with **True Multi-GPU Parallel Utilization**!

Since AudiobookMaker uses Python, multi-GPU pool dispatch, and a high-performance Rust PyO3 acceleration module (`audiobook_rust`), this notebook will:
1. Validate environment configuration (Multi-GPU & Internet access).
2. Clone/locate the project repository.
3. Install the Rust compiler toolchain.
4. Compile the Rust PyO3 modules on the Kaggle Linux environment.
5. Install all Python dependencies (including PyTorch with CUDA & `qwen-tts`; other engines are optional, one per session).
6. Launch the FastAPI orchestration server in the background.
7. Launch the Gradio Web UI with a public shareable link.

### ⚡ Select GPU Runtime & Enable Internet (CRITICAL)
Before running any cells, make sure your Kaggle session is configured properly:
- Open the **Right Sidebar** -> **Session options**.
- Set **Accelerator** to **GPU T4x2** (recommended: auto-detects and uses both GPUs in parallel for 2x faster synthesis).
- Turn **Internet** toggle to **ON** (Required to download models, clone the repository, and install dependencies).

In [ ]:
# Check if GPU accelerator is assigned
import subprocess
try:
    gpu_info = subprocess.check_output("nvidia-smi").decode("utf-8")
    print(gpu_info)
    print("✅ GPU Accelerator is correctly assigned!")
except FileNotFoundError:
    print("\n" + "="*80)
    print("⚠️ WARNING: GPU accelerator not detected (nvidia-smi not found)!")
    print("The application will run, but text-to-speech generation will be EXTREMELY slow.")
    print("To enable GPU:")
    print("  1. In the right panel of the Kaggle notebook editor, expand 'Session options'.")
    print("  2. Change 'Accelerator' from 'None' to 'GPU T4x2' (or 'GPU P100').")
    print("="*80 + "\n")

### 🌐 Verify Internet Access
This cell checks if internet access is active. Kaggle notebooks disable internet by default, which will prevent cloning the repository or downloading required models.

In [ ]:
import socket

print("Checking internet connectivity...")
try:
    # Try to connect to github.com on port 80 to check if internet is enabled
    socket.setdefaulttimeout(5)
    socket.socket(socket.AF_INET, socket.SOCK_STREAM).connect(("github.com", 80))
    print("✅ Internet access is enabled! You are ready to download dependencies and models.")
except Exception as e:
    print("\n" + "="*80)
    print("❌ CRITICAL ERROR: Internet is not enabled in your Kaggle session!")
    print("We cannot clone the repository or download TTS models without internet access.")
    print("To enable Internet:")
    print("  1. In the right panel of the Kaggle notebook editor, expand 'Session options'.")
    print("  2. Toggle 'Internet' to ON.")
    print("  Note: If you don't see this option, you must verify your phone number in your Kaggle settings.")
    print("="*80 + "\n")
    raise RuntimeError("Internet connection is required but unavailable. Please enable Internet in Session options.")

### 📂 Clone Repository or Setup Workspace

In [ ]:
import os

# Kaggle's writable directory is /kaggle/working
KAGGLE_WORKING = '/kaggle/working'
os.chdir(KAGGLE_WORKING)

# Check if we are already in a repository root
if os.path.exists("audiobook_rust") and os.path.exists("requirements.txt"):
    print("Already in the AudiobookMaker repository root directory:", os.getcwd())
elif os.path.exists("AudiobookMaker/audiobook_rust"):
    print("Found AudiobookMaker directory. Changing directory...")
    os.chdir("AudiobookMaker")
    print("Current directory:", os.getcwd())
else:
    print("Cloning AudiobookMaker repository...")
    # Run git clone. If this fails, we will know immediately.
    clone_status = os.system("git clone https://github.com/MSpider3/AudiobookMaker.git")
    if clone_status != 0:
        raise RuntimeError("Failed to clone the repository! Please double-check your internet connection.")
    
    if os.path.exists("AudiobookMaker"):
        os.chdir("AudiobookMaker")
        print("Successfully cloned and entered AudiobookMaker directory.")
    else:
        raise FileNotFoundError("Could not find the cloned AudiobookMaker directory!")

# Final check
if not os.path.exists("audiobook_rust"):
    raise FileNotFoundError("Invalid workspace: 'audiobook_rust' directory not found at " + os.getcwd())

### 🦀 Install Rust Compiler Toolchain

In [ ]:
import os
import sys

print("Installing Rustup and Cargo...")
install_rust = os.system("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y")
if install_rust != 0:
    print("Warning: Rust installation script exited with non-zero status.")

# Add Cargo binaries to Python PATH env variable
cargo_bin = os.path.expanduser("~/.cargo/bin")
if cargo_bin not in os.environ["PATH"]:
    os.environ["PATH"] += os.pathsep + cargo_bin

# Verify Rust tools are available
print("\nVerifying Rust compiler installation:")
cargo_ver = os.system("cargo --version")
rustc_ver = os.system("rustc --version")

if cargo_ver != 0 or rustc_ver != 0:
    print("⚠️ WARNING: Rust compiler or Cargo not found in PATH.")
    print("Make sure you add ~/.cargo/bin to your PATH.")


### 📦 Install System Dependencies & Python Requirements

In [ ]:
import os
import sys

print("Updating system packages and installing FFmpeg, SoX & OpenSSH...")
os.system("apt-get update && apt-get install -y ffmpeg sox libsox-fmt-all openssh-client")

print("\nInstalling Python dependencies (this will take a few minutes)...")
pip_status = os.system(f"{sys.executable} -m pip install -r requirements.txt")
if pip_status != 0:
    print("Warning: pip install -r requirements.txt returned a non-zero status.")

print("\nInstalling maturin (needed to build python-rust PyO3 modules)...")
os.system(f"{sys.executable} -m pip install maturin")


### ⚙️ Compile Rust PyO3 Module

In [ ]:
import os

print("Compiling audiobook_rust module using maturin...")
current_dir = os.getcwd()
try:
    os.chdir('audiobook_rust')
    build_status = os.system("pip install .")
    if build_status != 0:
        print("Warning: maturin build returned a non-zero status.")
finally:
    os.chdir(current_dir)

# Verify import works
try:
    import importlib
    importlib.invalidate_caches()
    import audiobook_rust
    print("🎉 Success! audiobook_rust compiled and imported successfully. 5.5x faster mastering is enabled.")
except ImportError as e:
    print("\n" + "="*80)
    print("⚠️ WARNING: Error compiling Rust code:", e)
    print("Fallback: The app will run, but with slower pure-Python fallback code.")
    print("="*80 + "\n")

### 🎙️ (Optional) Use Another TTS Engine

**Qwen3-TTS** is installed and ready. To narrate with another engine, set `ENGINE` in the cell below and run it.

| `ENGINE` | Engine | Weights licence | Good to know |
|---|---|---|---|
| `qwen` | Qwen3-TTS (default) | Apache-2.0 | 10 languages; fastest on a T4 |
| `higgs` | Higgs Audio v3 | non-commercial | Fast; the engine to use for Hindi |
| `omnivoice` | OmniVoice | non-commercial | 600+ languages, small |
| `moss` | MOSS-TTS | Apache-2.0 | ~13 GB download |
| `indextts` | IndexTTS-2.5 | bilibili licence | Emotion control |
| `fish` | Fish Audio S2 Pro | non-commercial | Needs the clip's transcript; slow on a T4 |
| `f5tts` | F5-TTS | non-commercial | English / Chinese |

**One engine per session.** Each engine needs its own version of the `transformers` library, and installing one replaces the version Qwen3-TTS needs. To switch back, restart the session and leave `ENGINE = "qwen"`.

In [ ]:
import os
import sys

ENGINE = "qwen"   # "qwen", "higgs", "omnivoice", "moss", "indextts", "fish" or "f5tts"

# Second install step of the engines whose own package pins libraries the notebook must not change.
EXTRA_STEPS = {
    "indextts": '--no-deps --ignore-requires-python descript-audiotools '
                '"indextts @ git+https://github.com/index-tts/index-tts.git@d9e41aac89fd00b3d71497fddb287b7f24613712"',
    "fish": '--no-deps descript-audiotools descript-audio-codec '
            '"fish-speech @ git+https://github.com/fishaudio/fish-speech.git@214da3cd841bda85da2496b96cd3c4d7edb1337e"',
}

if ENGINE == "qwen":
    print("✅ Qwen3-TTS is already installed. Nothing to do.")
elif ENGINE == "f5tts":
    status = os.system(f"{sys.executable} -m pip install -q f5-tts")
    print("✅ F5-TTS installed. Select 'f5tts' in the UI." if status == 0 else "⚠️ F5-TTS installation failed.")
elif os.path.exists(f"requirements/tts-{ENGINE}.txt"):
    print(f"Installing {ENGINE} (a few minutes)...")
    status = os.system(f"{sys.executable} -m pip install -q -r requirements/tts-{ENGINE}.txt")
    if status == 0 and ENGINE in EXTRA_STEPS:
        status = os.system(f"{sys.executable} -m pip install -q {EXTRA_STEPS[ENGINE]}")
    if ENGINE == "indextts":
        os.environ["USE_MODELSCOPE"] = "false"   # download from Hugging Face
    if status == 0:
        print(f"✅ {ENGINE} installed. Select it as TTS Provider in the UI, or pass --provider {ENGINE} to cli.py.")
        print("   Qwen3-TTS will not load in this session any more; restart the session to use it again.")
        os.system(f"{sys.executable} cli.py --list-providers | head -60")
    else:
        print(f"⚠️ Installing {ENGINE} failed - see the messages above.")
else:
    print(f"⚠️ Unknown engine {ENGINE!r}. Choose one of: qwen, higgs, omnivoice, moss, indextts, fish, f5tts.")


### ⚡ (Optional) Install FlashAttention-2
FlashAttention-2 provides a massive speedup and memory reduction for Qwen3-TTS. This can take several minutes to compile depending on the environment.

In [ ]:
import os
import sys

print("Installing flash-attn (this may take a while)...")
res = os.system(f"{sys.executable} -m pip install flash-attn --no-build-isolation")
if res == 0:
    print("\n✅ flash-attn installed successfully!")
else:
    print("\n⚠️ Warning: flash-attn installation failed or skipped.")


### 🔍 Pre-Run Diagnostics & Health Check (Optional)
Run this cell anytime to verify dual GPU memory, module imports, Rust acceleration, and Multi-GPU pool dispatch without loading the heavy 1.7B TTS model.

In [ ]:
import os
if os.path.exists("scripts/kaggle_prerun_check.py"):
    os.system("python scripts/kaggle_prerun_check.py")
else:
    from audiobook_factory.gpu_pool import GPUDetector
    print("GPU Devices:", GPUDetector.detect_devices())


### 🚀 Start FastAPI Orchestration Server in Background

In [ ]:
import subprocess
import time
import requests
import sys

print("Starting start_api.py in the background...")
api_proc = subprocess.Popen([sys.executable, "start_api.py"])

print("Waiting for FastAPI server to start...")
for i in range(45):
    time.sleep(1)
    if api_proc.poll() is not None:
        print(f"\n⚠️ start_api.py exited early with code {api_proc.returncode}. Using local processing fallback.")
        break
    try:
        r = requests.get("http://127.0.0.1:8000/api/v1/health", timeout=1.0)
        if r.status_code == 200 and r.json().get("status") == "ok":
            print(f"\n✅ FastAPI Server is healthy and running after {i+1} seconds!")
            break
    except Exception:
        print(".", end="", flush=True)
else:
    print("\n⚠️ Warning: FastAPI healthcheck timed out. Python Gradio UI will use local processing fallback.")


### 🎨 Launch Gradio Web Interface & Tunnel
Run this cell to start the UI. This cell will:
1. Start the Gradio Web UI locally on the Kaggle environment.
2. Establish a **public tunnel** using Pinggy (which is highly reliable on Kaggle and requires no account or setup).

In [ ]:
from app import build_app
import subprocess
import time
import os
import re
import threading
import logging
import sys

# Configure logging: detailed info for AudiobookMaker while suppressing noisy 3rd-party trace logs
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logging.getLogger("audiobook_factory").setLevel(logging.DEBUG)
for noisy_logger in ["urllib3", "httpcore", "httpx", "pydot", "asyncio", "multipart", "gradio", "uvicorn"]:
    logging.getLogger(noisy_logger).setLevel(logging.WARNING)

# ── 1. Ensure SSH Key Exists for Pinggy Tunnel (Bypasses Password Prompts) ────
ssh_dir = os.path.expanduser("~/.ssh")
os.makedirs(ssh_dir, exist_ok=True)
try:
    os.chmod(ssh_dir, 0o700)
except Exception:
    pass
key_path = os.path.join(ssh_dir, "id_ed25519")
if not os.path.exists(key_path):
    print("Generating ephemeral SSH key for secure Pinggy tunnel...")
    subprocess.run(["ssh-keygen", "-t", "ed25519", "-N", "", "-f", key_path, "-q"], check=True)
try:
    os.chmod(key_path, 0o600)
except Exception:
    pass

# ── 2. Start Pinggy Public Tunnel in Background (Auto-renewing) ───────────────
RENEW_INTERVAL_SECONDS = 50 * 60
SSH_CMD = [
    "ssh", "-p", "443",
    "-R", "0:localhost:7860",
    "-i", key_path,
    "-o", "BatchMode=yes",
    "-o", "StrictHostKeyChecking=no",
    "-o", "UserKnownHostsFile=/dev/null",
    "-o", "LogLevel=ERROR",
    "-o", "ServerAliveInterval=30",
    "-T", "a@free.pinggy.io"
]
URL_RE = re.compile(r"https?://[a-zA-Z0-9_.-]+pinggy[a-zA-Z0-9_.-]+")
_tunnel_stop = threading.Event()
public_urls = []

def _start_tunnel():
    """Start one SSH tunnel process and return (proc, urls_list, reader_thread)."""
    proc = subprocess.Popen(SSH_CMD, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    urls = []
    def _reader(p, u):
        try:
            for line in iter(p.stdout.readline, ""):
                if not line:
                    break
                if "pinggy" in line and "dashboard" not in line and "admin" not in line:
                    m = URL_RE.search(line)
                    if m:
                        link = m.group(0)
                        if link.startswith("http://"):
                            link = "https://" + link[7:]
                        if link not in u:
                            u.append(link)
        except Exception:
            pass
    t = threading.Thread(target=_reader, args=(proc, urls), daemon=True)
    t.start()
    return proc, urls, t

def _tunnel_manager():
    generation = 0
    print("🚀 Starting background Pinggy tunnel...")
    while not _tunnel_stop.is_set():
        generation += 1
        try:
            proc, urls, _ = _start_tunnel()
        except FileNotFoundError:
            print("⚠️ 'ssh' command not found. Please install openssh-client.")
            return
        except Exception as exc:
            print(f"⚠️ Error starting Pinggy tunnel: {exc}")
            return

        # Wait up to 25s for the URL to be captured
        for _ in range(25):
            if urls or _tunnel_stop.is_set() or proc.poll() is not None:
                break
            time.sleep(1)

        if urls:
            new_url = urls[0]
            if new_url not in public_urls:
                public_urls.append(new_url)
            print("\n" + "=" * 80)
            print(f"👉 PUBLIC TUNNEL LINK (Pinggy): {new_url}")
            print("=" * 80 + "\n")
        elif proc.poll() is not None:
            print(f"⚠️ Pinggy SSH tunnel exited with code {proc.returncode}. Retrying in 5s...")
            time.sleep(5)
            continue

        _tunnel_stop.wait(timeout=RENEW_INTERVAL_SECONDS)
        try:
            proc.terminate()
            proc.wait(timeout=5)
        except Exception:
            pass

manager_thread = threading.Thread(target=_tunnel_manager, daemon=True)
manager_thread.start()

# ── 3. Build & Launch Gradio Web Interface ────────────────────────────────────
demo = build_app()

print("⚡ Launching Gradio server...")
try:
    app_obj, local_url, share_url = demo.launch(
        server_name="0.0.0.0",
        server_port=7860,
        inline=False,             # Prevents infinite loop while checking url_ok in notebooks
        share=True,               # Attempt official gradio.live public link
        prevent_thread_lock=True, # Allows main thread to manage keepalive & tunnel
        debug=False,
    )
    print(f"✅ Gradio local server running at {local_url}")
    if share_url:
        if share_url not in public_urls:
            public_urls.append(share_url)
        print("\n" + "=" * 80)
        print(f"🎉 OFFICIAL PUBLIC GRADIO LINK: {share_url}")
        print("=" * 80 + "\n")
except Exception as e:
    print(f"⚠️ Gradio share link failed ({e}); falling back to local port 7860 with Pinggy tunnel...")
    try:
        demo.launch(
            server_name="0.0.0.0",
            server_port=7860,
            inline=False,
            share=False,
            prevent_thread_lock=True,
            debug=False,
        )
        print("✅ Gradio local server running on port 7860 (accessible via Pinggy link).")
    except Exception as e2:
        print(f"❌ Failed to launch Gradio: {e2}")

print("\n🚀 Web UI is active! Click the link(s) above to open.")
print("Press 'Stop' in your notebook when finished to terminate.\n")

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\nStopping web server and tunnel...")
    _tunnel_stop.set()
    manager_thread.join(timeout=5)
    demo.close()


### ⚡ Headless CLI Generation (Alternative to Gradio)

**Prefer the CLI for faster, more reliable generation — especially on long sessions where the Gradio tunnel may expire.**

#### Workflow:
1. Use the **Gradio cell above** to configure all settings (voice, chapters, TTS model, etc.).
2. In the Gradio UI, click **'📋 Export Config JSON'** to save your settings and all chapter text into a single `generation_progress.json` file.
3. Download the JSON file from Gradio (or find it in `audiobook_output/<BookTitle>/generation_progress.json`).
4. Fill in the paths/links below and run this cell.

**Supported input sources & overrides:**
- Upload files via Kaggle **Add data** sidebar → **Upload**.
- Paste a Google Drive shareable link (auto-downloaded via `gdown`).
- Leave `BOOK_FILE` / `VOICE_FILE` blank if JSON already has cached chapter text (most common for resumes).
- Optionally set `OUTPUT_DIR`, `OUTPUT_FMT` (`mp3`, `m4b`, `flac`, `wav`), or `QUANTIZATION` (`int8`, `none`).

**New in this version**
- Leave `CONFIG_JSON` empty and fill `BOOK_FILE` to narrate **straight from the book** with default settings - no Gradio step needed.
- `ENGINE`, `LANGUAGE`, `VOICE_TRANSCRIPT`, `CHAPTERS`, `SINGLE_FILE` and `PARALLEL_MODE` override what the JSON says. `PARALLEL_MODE = "auto"` is worth trying with two GPUs when the book has short chapters.


In [ ]:
import os, subprocess, sys

# ── Fill in your file paths or Google Drive / HTTP links ─────────────────────
CONFIG_JSON  = ""  # Path or link to generation_progress.json (leave empty to narrate BOOK_FILE directly)
BOOK_FILE    = ""  # Optional: path or link to book file (.epub, .pdf, etc.)
VOICE_FILE   = ""  # Path or link to narrator voice (.wav); required unless the JSON already has one
COVER_FILE   = ""  # Optional: path or link to cover image (.jpeg, .jpg, .png, .webp)
OUTPUT_DIR   = ""  # Optional: override output directory (e.g. /kaggle/working/audiobook_output)
OUTPUT_FMT   = ""  # Optional: override format ("mp3", "m4b", "flac", "wav")
QUANTIZATION = ""  # Optional: "none" or "int8" (int8 reduces VRAM by ~50%)
ENGINE       = ""  # Optional: TTS engine ("qwen", "higgs", ...) - install it in the engine cell first
LANGUAGE     = ""  # Optional: language of the book, e.g. "English", "Hindi", "Japanese"
VOICE_TRANSCRIPT = ""  # Optional: the exact words spoken in the voice clip, or a path to a .txt file with them
CHAPTERS     = ""  # Optional: only these chapter numbers, e.g. "1-5" or "1,3,7"
SINGLE_FILE  = False  # True: one audiobook file with chapter markers (use OUTPUT_FMT = "m4b")
PARALLEL_MODE = ""  # Optional: "chunks" (default), "chapters" or "auto" - how several GPUs are used

# ── Auto-download helper ──────────────────────────────────────────────────────
def resolve_file(path_or_url, default_name="file"):
    if not path_or_url:
        return ""
    if "drive.google.com" in path_or_url:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown
        dl = gdown.download(path_or_url, quiet=False, fuzzy=True)
        if dl and os.path.exists(dl):
            return os.path.abspath(dl)
        out = f"/kaggle/working/{default_name}"
        return out if os.path.exists(out) else path_or_url
    elif path_or_url.startswith("http"):
        import urllib.parse
        parsed = urllib.parse.urlparse(path_or_url)
        filename = os.path.basename(parsed.path) or default_name
        out = f"/kaggle/working/{filename}"
        subprocess.run(["wget", "-q", "-O", out, path_or_url], check=True)
        return out
    return path_or_url

config_path = resolve_file(CONFIG_JSON, "generation_progress.json")
book_path   = resolve_file(BOOK_FILE,   "book_file")
voice_path  = resolve_file(VOICE_FILE,  "narrator_voice.wav")
cover_path  = resolve_file(COVER_FILE,  "cover_image.jpg")

have_config = bool(config_path) and os.path.exists(config_path)
if not have_config and not book_path:
    raise FileNotFoundError("Fill in CONFIG_JSON (a generation_progress.json) or BOOK_FILE above.")

# ── Build and run CLI command ─────────────────────────────────────────────────
# --local: run in this cell, so the log shows here even when the API server is up.
if have_config:
    cmd = f'"{sys.executable}" cli.py "{config_path}" --local'
    if book_path:
        cmd += f' --book-path "{book_path}"'
else:
    cmd = f'"{sys.executable}" cli.py --book "{book_path}" --local'
if voice_path:
    cmd += f' --voice-file "{voice_path}"'
if cover_path:
    cmd += f' --cover-image "{cover_path}"'
if OUTPUT_DIR:
    cmd += f' --output-dir "{OUTPUT_DIR}"'
if OUTPUT_FMT:
    cmd += f' --output-format "{OUTPUT_FMT}"'
if QUANTIZATION:
    cmd += f' --quantization "{QUANTIZATION}"'
if ENGINE:
    cmd += f' --provider "{ENGINE}"'
if LANGUAGE:
    cmd += f' --language "{LANGUAGE}"'
if VOICE_TRANSCRIPT:
    flag = "--voice-transcript-file" if os.path.isfile(VOICE_TRANSCRIPT) else "--voice-transcript"
    cmd += f' {flag} "{VOICE_TRANSCRIPT}"'
if CHAPTERS:
    cmd += f' --chapters "{CHAPTERS}"'
if SINGLE_FILE:
    cmd += ' --single-file'
if PARALLEL_MODE:
    cmd += f' --parallel-mode "{PARALLEL_MODE}"'

print(f"Running: {cmd}\n")
os.system(cmd)
